In [1]:
import numpy as np 
import matplotlib.pyplot as plt 

import hyperspy.api as hs
import lumispy as lum

# Some environments (like the linked Binder) cannot support the Qt interface, so it may not work properly in those cases.
# Consider using alternative backends like %matplotlib inline or %matplotlib notebook for compatibility in such environments.
# However, note that some of the presented interactive functionalities will not work with `inline` plotting.
%matplotlib qt

# Batch spectra processing

#### Loading files


HyperSpy supports a wide range of microscopy and spectroscopy related **data file types**:
- Horiba JobinYvon XML `.xml`
- TriVista XML `.tvf`
- Renishaw wire format `.wdf`
- Gatan `.dm3/.dm4`
- Attolight `.sur` (For the moment does not parse `original_metadata` to `metadata`)
- Delmic `.hdf5` 
- Hamamatsu streak camera images in `.tif` format (Reading the itex `.img` format will be available in RosettaSciIO)

For saving analyses, HyperSpy has its own hdf5-based data format `.hspy`.

*We assume the file location as in the demo repository, if you downloaded the notebook and the data files individually, you might need to adapt the path.*

We first load multiple cathodoluminescence spectra from a folder, and *stack* them on top of one another, loading them as one signal that we can process:

In [6]:
spectra = hs.load('data/CL_spec1*.hysp', stack = True) 

ValueError: No filename matches the pattern "data/CL_spec1*.hysp"

In [5]:
hs.plot.plot_spectra(spectra)

<Axes: xlabel='Wavelength (nm)', ylabel='Intensity'>

We can inspect the different axes of our signal by calling the `axes_manager`. For this data our spectra have a signal axis `Wavelength`, and a navigational axis where our spectra have been stacked.

In [4]:
spectra.axes_manager

Navigation axis name,size,index,offset,scale,units
stack_element,5,4,0.0,1.0,
Signal axis name,size,,offset,scale,units
Wavelength,1336,,346.41383699764265,0.20711804926395416,nm


#### Convert to Count/s
The spectra loaded were measured for different aquisition/exposure times, so we can compare their true intensities we convert them to Count/s.

In [6]:
spectra.scale_by_exposure(inplace = True)

Intensity (Counts/s)


In [8]:
hs.plot.plot_spectra(spectra)
plt.title('Spectra in Count/s')

Text(0.5, 1.0, 'Spectra in Count/s')

#### Jacobian transformation

To preserve the integrated intensity per spectral window, a *Jacobian* transformation has to be applied to the signal intensity. As we require $I(E)dE = I(\lambda)d\lambda$, the scale transformation $E=hc/\lambda$ implies

$$I(E) = I(\lambda)\frac{d\lambda}{dE} = I(\lambda)\frac{d}{dE}
\frac{h c}{E} = - I(\lambda) \frac{h c}{E^2}$$

![Alt Text](jacobian_transformation.png)

Converting between nm, eV and cm^-2 can be easily done with wrapper functions in LumiSpy, in this case we convert our axis to eV and inspect the changes:

In [ ]:
spectra.to_eV(inplace = True)
spectra.axes_manager

Navigation axis name,size,index,offset,scale,units
stack_element,5,4,0.0,1.0,
Signal axis name,size,,offset,scale,units
Energy,1336,,non-uniform axis,non-uniform axis,eV


`to_eV()` produces a non-uniform axis - one represented by an array since it has unequal spacing between values. In order to complete the next analysis step we're going to convert our axis to a uniform one.

In [10]:
spectra = spectra.interpolate_on_axis('uniform', axis='Energy')

#### Peak analysis
HyperSpy has several peak analysis wrapper functions for signals, such as `.find_peaks_ohaver`... For our data we compute the full width half maxima (FWHM) of our spectra:

In [13]:
fwhms = spectra.estimate_peak_width()
fwhms.data

  0%|          | 0/2 [00:00<?, ?it/s]

array([0.13814164, 0.14183938, 0.18568952, 0.1793272 , 0.23689516])

The centroid in the wavelength axis of each spectrum in our signal can be easily obtained:

In [16]:
centroids = spectra.centroid()
centroids.data

  0%|          | 0/2 [00:00<?, ?it/s]

array([2.93036152, 2.8976513 , 2.7300502 , 2.86256788, 2.80661054])

#### Plotting results
Next we can inspect our processed data, and also add our FWHM and centroid values to the legend.

In [21]:
hs.plot.plot_spectra(spectra, 
                     legend=[
        f"{c:.2f} eV, {f*1000:.3g} meV"
        for c, f in zip(centroids.data, fwhms.data)
    ])

<Axes: xlabel='Energy (eV)', ylabel='Intensity'>

Finally, we can save the changes we've made to our spectra.

In [ ]:
spectra.save('spectra_stack_scaled_eV.hspy')

# Spectral map *Hyperspectral* analysis

#### Load and explore data
*(In the following, we will use the dataset `spectra_map` saved in the Gatan format. The sample contains (In,Ga)N measured by Aidan Campbell at the Paul Drude Institute, Berlin.)*

In [24]:
spectra_map = hs.load('InGaN_hyperspectral.dm4').inav[13:,13:]

Our sample dataset has **two navigation dimensions** (60, 60 and **one signal (spectral) dimension** |1336). When we plot the data as it is, we can navigate the map across the 60, 60 spatial dimensions:

In [25]:
spectra_map.plot()

#### Clean cosmic rays from spectra
`.spikes_removal_tool` can automatically clean your data or can be used interactively where removal of each spike is confirmed by user 

In [27]:
spectra_map.spikes_removal_tool(interactive = False) 

#### Chromatic imaging

Indexing can also be used for color-filtered (chromatic) imaging. First, lets plot the map with its intensity summed across the wavelength axis:

*(the object is transposed, so that we plot the intensity over navigation instead of signal dimensions)*

In [30]:
summed_map = spectra_map.T.mean()
summed_map.plot(cmap='inferno')

Now, we can **plot the intensity at a selected spectral value or window** using indexing for intensity at λ=500 nm.

In [31]:
spectra_map.isig[500.0].T.mean().plot()

Alternatively, we can explore spectral map interactively by integrating over an adaptable region of interest (ROI).

In [32]:
rois, [map1, map2] = hs.plot.plot_roi_map(spectra_map, rois = 2, 
                     single_figure = True)

We can directly access the ROIs and the new maps we've extracted.

In [33]:
rois, [map1, map2]

([SpanROI(left=375.721, right=394.569), SpanROI(left=445.727, right=476.588)],
 [<Signal2D, title: Integrated intensity, dimensions: (|47, 47)>,
  <Signal2D, title: Integrated intensity, dimensions: (|47, 47)>])

#### Combined plot
Next we combine our results into one large combined interactive figure

In [42]:
# Declare matplotlib figure
fig = plt.figure(figsize = [7,6])
subfigs = fig.subfigures(2, 2, wspace=0.07)

# Declare our ROIs
roi1 = hs.roi.RectangularROI()
roi2 = hs.roi.RectangularROI()

# Plot subplots
spectra_map.plot(colorbar=False)
summed_map.plot(colorbar=False, fig = subfigs[0,0], cmap = 'inferno', title = 'Summed')
map1.plot(cmap = 'Oranges', fig=subfigs[1,0], colorbar=False, title= f"{rois[0][0]:.3g} - {rois[0][1]:.3g} nm")
map2.plot(cmap = 'Blues', fig=subfigs[1,1], colorbar=False, title= f"{rois[1][0]:.3g} - {rois[1][1]:.3g} nm")

# Attach ROIs to relevant plots
roi1.add_widget(spectra_map)
roi1.interactive(map2)
roi1.interactive(summed_map)

roi2.add_widget(spectra_map)
roi2.interactive(map1)
roi2.interactive(summed_map)

# Compute mean signal inside of ROIs
def roi_mean_func(s, roi):
    return roi(s).mean()

roi1_spectra_mean = hs.interactive(roi_mean_func, 
                                event=roi1.events.changed,
                                s = spectra_map,
                                roi = roi1)
roi2_spectra_mean = hs.interactive(roi_mean_func, 
                                event=roi2.events.changed,
                                s = spectra_map,
                                roi = roi2)

# Plot extracted mean signals
hs.plot.plot_spectra([roi1_spectra_mean, roi2_spectra_mean, spectra_map.mean()],
                     legend = ['ROI 1','ROI 2','Mean'],
                     colors = ['blue','orange','black'],
                     fig=subfigs[0,1],
                     normalise = True)
plt.tight_layout()

# Time-resolved luminescence analysis

*(In the following, we will use the preprocessed dataset `streak01` saved in the HyperSpy format. The sample contains an AlN layer measured by Domenik Spallek at the Paul Drude Insitute, Berlin.)*

The streak images are of the `LumiTransientSpectrum` signal class:

In [2]:
streak_image = hs.load('AlN_HR_streak_image.hspy')
streak_image.axes_manager

Signal axis name,size,,offset,scale,units
Wavelength,220,,204.7,0.008,nm
Time,217,,-80.0,2.4392644399322356,ps


We can interactively inspect our streak image and compare transients from the `sum` of vertical slices on the image using two ROIs:

In [ ]:
# Combine plots in one figure
fig = plt.figure(figsize = [8,5])
subfigs = fig.subfigures(2, 2, wspace=0.07, height_ratios=[1, 3])

# Declare ROIs
roi1 = hs.roi.RectangularROI()
roi2 = hs.roi.RectangularROI()

# Plot streak image
streak_image.plot(fig=subfigs[1,0], colorbar = False, 
              title = 'Streak image', cmap = 'turbo')

# Connect ROIs to plot
roi1.add_widget(streak_image, color="red")
roi2.add_widget(streak_image , color="blue")

# Declare function for meaning along wavelength axis
def roi_mean_function(s, roi):
    return roi(s).mean(axis='Wavelength')

# Compute function and interactivity
s_roi_mean1 = hs.interactive(
    roi_mean_function,
    event=roi1.events.changed,
    s=streak_image,
    roi=roi1,
)

s_roi_mean2 = hs.interactive(
    roi_mean_function,
    event=roi2.events.changed,
    s=streak_image,
    roi=roi2,
)
    
# Plot extracted transients and summed spectrum
hs.plot.plot_spectra([s_roi_mean1,
                      s_roi_mean2], 
                     color = ['red','blue'], 
                     normalise = True,
                     fig=subfigs[1,1])

hs.plot.plot_spectra([streak_image.sum(axis = 'Time')], 
    color = ['k'], 
    fig=subfigs[0,0])
    

<Axes: xlabel='Wavelength (nm)', ylabel='Intensity'>

#### More functions...


#### In progress...


In [ ]:
tau_eff1 = hs.interactive(
    get_tau_eff,
    event=roi1.events.changed, #.connect(update_legend),
    s=s_roi_mean1,
)

def update_fig(event=None):
    subfigs[1].text(0.7, 0.8, str(tau_eff1))
    print(tau_eff1)

def get_tau_eff(s):
    first_moment = (s * s.axes_manager['Time'].axis).integrate1D(axis = 0)
    zeroth_moment = s.integrate1D(axis = 0)
    return first_moment.data/zeroth_moment.data

#### _Retired cells


In [ ]:
transient1 = integrated_sliced_signal1

In [ ]:
get_tau_eff(integrated_sliced_signal2)

In [ ]:
# Combine plots in one figure
fig = plt.figure(figsize = [7,6])
subfigs = fig.subfigures(2, 2, wspace=0.07, height_ratios=[1, 3])

# Declare regions of interests (ROI)
roi1 = hs.roi.SpanROI(left = 204.8, right = 204.9)
roi2 = hs.roi.SpanROI(left = 205, right = 205.1)

# Plot streak image and add ROIs
streak01.plot(fig=subfigs[1, 0], colorbar = False, 
              title = 'Streak image', cmap = 'turbo')
sliced_signal1 = roi1.interactive(streak01, axes=streak01.axes_manager[0], color = 'blue')
sliced_signal2 = roi2.interactive(streak01, axes=streak01.axes_manager[0], color = 'red')

# Create new signals which are summed from ROI
integrated_sliced_signal1 = hs.interactive(
    sliced_signal1.sum,
    axis=sliced_signal1.axes_manager[0],
    event=roi1.events.changed,
    recompute_out_event=None)

integrated_sliced_signal2 = hs.interactive(
    sliced_signal2.sum,
    axis=sliced_signal2.axes_manager[0],
    event=roi2.events.changed,
    recompute_out_event=None)

# Plot extracted transients in new figure
hs.plot.plot_spectra([integrated_sliced_signal1,
                      integrated_sliced_signal2], 
                     color = ['blue','red'], 
                     normalise = True,
                     fig=subfigs[1,1])

hs.plot.plot_spectra([streak01.sum(axis = 'Time')], 
    color = ['k'], 
    fig=subfigs[0,0])

In [ ]:
roi = hs.roi.PolygonROI()

spectra_map.plot(colorbar=False)
map2.plot(cmap = 'Blues')

roi_spectra = roi.interactive(spectra_map)
roi.interactive(map2)

roi_spectra_mean = hs.interactive(roi_spectra.nanmean, 
                                event=roi.events.changed,
                                recompute_out_event=None)

spectra_map_mean = spectra_map.mean()

hs.plot.plot_spectra([spectra_map_mean, roi_spectra_mean],
                     legend = ['Map mean','ROI mean'])

In [ ]:
roi

In [ ]:
for i in range(len(spectra)):
    spectra[i].scale_by_exposure(inplace = True)

In [ ]:
# Declare matplotlib figure
fig = plt.figure(figsize = [7,7])
subfigs = fig.subfigures(2, 2, wspace=0.07)

# Declare our ROIs
roi1 = hs.roi.RectangularROI()
roi2 = hs.roi.RectangularROI()

# Plot subplots
spectra_map.plot(colorbar=False)
summed_map.plot(colorbar=False, fig = subfigs[0,0], cmap = 'inferno')
map1.plot(cmap = 'Oranges', fig=subfigs[1,0], colorbar=False)
map2.plot(cmap = 'Blues', fig=subfigs[1,1], colorbar=False)

# Attach ROIs to relevant plots
roi1_spectra = roi1.interactive(spectra_map)
roi1.interactive(map2)
roi1.interactive(summed_map)

roi2_spectra = roi2.interactive(spectra_map)
roi2.interactive(map1)
roi2.interactive(summed_map)

def roi_mean_function(s, roi):
    return roi(s).mean(axis='Wavelength')

# Compute mean signal inside of ROIs
roi1_spectra_mean = hs.interactive(roi1_spectra.nanmean, 
                                event=roi1.events.changed,
                                recompute_out_event=None)
roi2_spectra_mean = hs.interactive(roi2_spectra.nanmean, 
                                event=roi2.events.changed,
                                recompute_out_event=None)

# Plot extracted mean signals
hs.plot.plot_spectra([roi1_spectra_mean, roi2_spectra_mean, spectra_map.mean()],
                     legend = ['ROI 1','ROI 2','Mean spectrum'],
                     fig=subfigs[0,1],
                     normalise = True)